<div style="
    padding: 3px;
    background: linear-gradient(135deg, #667eea, #764ba2, #f093fb);
    border-radius: 12px;
    display: inline-block;
    width: 100%;
    box-sizing: border-box;
">
    <div style="
        background: #1e1e1e;
        border-radius: 10px;
        padding: 20px 25px;
    ">
        <h1 style="color: white; margin: 0 0 8px 0;">E-Commerce Sales Analysis</h1>
        <p style="color: #aaa; margin: 4px 0;"><b style="color: white;">Dataset:</b> Online Retail UK (Kaggle)</p>
        <p style="color: #aaa; margin: 4px 0;"><b style="color: white;">Goal:</b> Data preprocessing for further SQL analysis and Power BI visualization</p>
    </div>
</div>


### **[Dataset](https://www.kaggle.com/datasets/carrie1/ecommerce-data?select=data.csv)**

************

## Libraries import / Data Loading :

Loading the raw dataset with `latin-1` encoding due to special characters in product descriptions.

In [1]:
import pandas as pd

In [2]:
ecommerce = pd.read_csv("ecommercedatab4preproc.csv" , encoding = "latin-1")

ecommerce.head(10).sort_values(by="Quantity" , ascending = False)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom


## 2. Exploratory Data Analysis :
Initial overview of data types, missing values and basic statistics.

In [15]:
print(f"{ecommerce.describe()} \n , {ecommerce.dtypes}")

            Quantity                    InvoiceDate      UnitPrice  \
count  537382.000000                         537382  537382.000000   
mean        9.584856  2011-07-04 16:53:27.756679168       3.690658   
min    -80995.000000            2010-12-01 08:26:00       0.001000   
25%         1.000000            2011-03-28 11:58:00       1.250000   
50%         3.000000            2011-07-20 11:55:00       2.080000   
75%        10.000000            2011-10-19 11:49:00       4.130000   
max      4800.000000            2011-12-09 12:50:00    4505.170000   
std       155.267240                            NaN      17.434705   

          InvoiceDay   InvoiceMonth    InvoiceYear    InvoiceHour  \
count  537382.000000  537382.000000  537382.000000  537382.000000   
mean       15.016290       7.556894    2010.921689      13.078936   
min         1.000000       1.000000    2010.000000       6.000000   
25%         7.000000       5.000000    2011.000000      11.000000   
50%        15.000000    

## 3. Data Cleaning
### 3.1 Removing invalid UnitPrice
Filtering out rows where `UnitPrice` ≤ 0 as they represent system entries, not real transactions.

### 3.2 Removing service transactions
Dropping non-product `StockCodes`: Amazon fees, postage, manual adjustments.

### 3.3 Outliers
Removing two invoices with unrealistic quantities (80k+ units).

In [4]:
ecommerce = ecommerce[ecommerce['UnitPrice'] > 0]

In [5]:
quantity_outliers = ecommerce[ecommerce['Quantity'] > 10000]

unitprice_outliers = ecommerce[ecommerce['UnitPrice'] > 5000]

In [6]:
service_codes = ['AMAZONFEE', 'POST', 'M', 'B', 'D', 'S', 'CRUK']

ecommerce[ecommerce['StockCode'].isin(service_codes)].value_counts('StockCode')


StockCode
POST         1252
M             565
D              77
S              63
AMAZONFEE      34
CRUK           16
B               1
Name: count, dtype: int64

In [7]:
ecommerce = ecommerce[~ecommerce['StockCode'].isin(service_codes)]

ecommerce = ecommerce[~ecommerce['InvoiceNo'].isin(['541431','581483'])]

In [8]:
ecommerce.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132343
Country             0
dtype: int64

### 3.4 Missing Values:
`CustomerID` NaN replaced with 'Guest' — these are anonymous purchases (~25% of transactions).

In [9]:
ecommerce['CustomerID'] = ecommerce['CustomerID'].fillna('Guest')

## 4. Feature Engineering:
Extracting date components from `InvoiceDate` and calculating `TotalPrice`.

In [10]:
ecommerce['CustomerID'] = ecommerce['CustomerID'].astype('object')

ecommerce['InvoiceDate'] = pd.to_datetime(ecommerce['InvoiceDate'])

ecommerce['Date'] = ecommerce['InvoiceDate'].dt.date
ecommerce['InvoiceTime'] = ecommerce['InvoiceDate'].dt.time
ecommerce['InvoiceDay'] = ecommerce['InvoiceDate'].dt.day
ecommerce['InvoiceMonth'] = ecommerce['InvoiceDate'].dt.month
ecommerce['InvoiceYear'] = ecommerce['InvoiceDate'].dt.year
ecommerce['DayOfWeek'] = ecommerce['InvoiceDate'].dt.day_name()
ecommerce['InvoiceHour'] = ecommerce['InvoiceDate'].dt.hour


In [14]:
ecommerce['TotalPrice'] = ecommerce['Quantity'] * ecommerce['UnitPrice'] 

In [13]:
ecommerce.head().sort_values(by='InvoiceTime' , ascending = False)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Date,InvoiceTime,InvoiceDay,InvoiceMonth,InvoiceYear,DayOfWeek,InvoiceHour,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010-12-01,08:26:00,1,12,2010,Wednesday,8,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,08:26:00,1,12,2010,Wednesday,8,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010-12-01,08:26:00,1,12,2010,Wednesday,8,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,08:26:00,1,12,2010,Wednesday,8,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010-12-01,08:26:00,1,12,2010,Wednesday,8,20.34


## 5. Export:
Saving cleaned dataset for SQL analysis.

In [17]:
ecommerce.to_csv('C:\\Users\\lelik\\Desktop\\vs\\AlexIrnazarovPet_projects\\E-commerceAnalysis\\Data\\ecommerce_cleaned.csv', index=False)